In [ ]:
import pandas as pd
import numpy as np
import itertools
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
import dalex as dx

from funs import (
    dataPreparation,
    createMetaDictionary,
    createDictionary,
    evaluateModel,
    dictionaryModel,
)

# Data preparation

## Base data

In [ ]:
data = dataPreparation(
    all_trxns_path="../data/all_trxns.csv", exchange_rates_path="../data/exchange_rates.csv"
)

## Train Test split

In [ ]:
data["fraud_flag_trans"] = data["fraud_flag"].replace({"N": 0, "Y": 1})
y = data["fraud_flag_trans"]
# Drop the synthetic 0/1 target BEFORE building X so the feature matrix
# physically cannot contain the label (label leakage guard). The original
# `fraud_flag` (Y/N string) is intentionally KEPT on X_train/X_test: it is not
# a model feature but is required by `createDictionary` / `dictionaryModel`
# (funs.py) to compute fraud-probability dicts and `fraud_flag_transformed`;
# it is dropped from the XGB input frame in the dictionary-model cells below.
X = data.drop(columns=["fraud_flag_trans"]).copy()

# Stratified split preserves the ~1.7% positive rate in both folds. A
# chronological split would risk leaving the test set with too few (or zero)
# frauds at this imbalance level; stratification is the safer default here.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## Train dictionaries

In [ ]:
dictionaries_to_get = [
    "customer_country",
    "counterparty_country",
    "type",
    "ccy",
    "customer_type",
    "weekday",
    "month",
    "quarter",
    "hour",
    "amount_eur_bucket",
]

dictionaries = {}
for dict_name in dictionaries_to_get:
    dictionaries[dict_name] = createDictionary(
        X_train, colname=dict_name, count_filter=0
    )

meta_dicts = []  # Use a list to collect dictionaries
for dict_name in dictionaries_to_get:
    meta_dict = createMetaDictionary(X_train, colname=dict_name, quantile_threshold=0.9, count_filter=0)
    meta_dicts.append(meta_dict)
meta_dictionary = pd.concat(meta_dicts, ignore_index=True)

## Dictionary Model train data

The dictionary thresholds are found by a principled grid search (maximizing F1
on the train set), the same method as notebook 4. Note: `predicted_fraud` is
dropped below because the dictionary model serves here as a feature engine for
XGBoost — the thresholds are fit for methodological consistency, but the final
prediction is made by XGBoost over the aggregation columns.

In [ ]:
# Principled threshold search on the train set (maximizing F1), matching the
# method in notebook 4. The permissive-threshold call returns the aggregation
# columns; the grid then scans the three-threshold space.
_train_agg = dictionaryModel(
    X_train, dictionaries, meta_dictionary,
    fraud_probability_threshold=0.0, sd_flags_threshold=-1, quantile_flags_threshold=-1,
)
_y_train_dict = _train_agg["fraud_flag_transformed"].values
_fp_grid = np.arange(0.05, 0.40, 0.025)
_flag_grid = [0, 1, 2, 3]
_best_f1_dict = -1.0
_best_dict_thresholds = None
for _fp_t, _sd_t, _qf_t in itertools.product(_fp_grid, _flag_grid, _flag_grid):
    _pred = (
        (_train_agg["expected_fraud_probability"] > _fp_t)
        & (_train_agg["sd_flags"] > _sd_t)
        & (_train_agg["quantile_flags"] > _qf_t)
    ).astype(int)
    _f1 = f1_score(_y_train_dict, _pred, zero_division=0)
    if _f1 > _best_f1_dict:
        _best_f1_dict = _f1
        _best_dict_thresholds = (float(_fp_t), int(_sd_t), int(_qf_t))
_dict_fp, _dict_sd, _dict_qf = _best_dict_thresholds
print(
    "Dictionary thresholds (F1=%.4f on train): "
    "expected_fraud_probability>%.3f, sd_flags>%d, quantile_flags>%d"
    % (_best_f1_dict, _dict_fp, _dict_sd, _dict_qf)
)

LP_model_train = dictionaryModel(
    X_train,
    dictionaries,
    meta_dictionary,
    fraud_probability_threshold=_dict_fp,
    sd_flags_threshold=_dict_sd,
    quantile_flags_threshold=_dict_qf,
)
LP_model_train = LP_model_train.drop(
    columns=["fraud_flag_transformed", "fraud_flag", "predicted_fraud"], axis=1
)

## Dictionary Model test data

In [ ]:
LP_model_test = dictionaryModel(
    X_test,
    dictionaries,
    meta_dictionary,
    fraud_probability_threshold=_dict_fp,
    sd_flags_threshold=_dict_sd,
    quantile_flags_threshold=_dict_qf,
)
LP_model_test = LP_model_test.drop(
    columns=["fraud_flag_transformed", "fraud_flag", "predicted_fraud"], axis=1
)

## One-hot encoding

In [ ]:
model_train_data = pd.get_dummies(LP_model_train, columns=dictionaries_to_get)
model_test_data = pd.get_dummies(LP_model_test, columns=dictionaries_to_get)

# Cast only object columns to float so XGBoost DMatrix accepts them. Datetime
# columns (timestamp) and string-ID columns (customer/counterparty) are left as-is
# and dropped in the next cell before they reach the model; the previous whole-
# DataFrame .astype(float) failed on DatetimeArray, and a blanket object-cast
# also fails on the non-numeric ID strings.
_DROP_COLS = {"customer", "timestamp", "counterparty"}
for _df in (model_train_data, model_test_data):
    for _col in _df.select_dtypes(include="object").columns:
        if _col in _DROP_COLS:
            continue
        _df[_col] = _df[_col].astype(float)

## Additional cleaning

In [ ]:
model_train_data.columns = [
    col.replace("[", "").replace("]", "") for col in model_train_data.columns
]
model_test_data.columns = [
    col.replace("[", "").replace("]", "") for col in model_test_data.columns
]

model_train_data.drop(["customer", "timestamp", "counterparty"], axis=1, inplace=True)
model_test_data.drop(["customer", "timestamp", "counterparty"], axis=1, inplace=True)

# XGBoost model

In [ ]:
print(model_train_data)

## Training

Grid search uses `average_precision` (PR-AUC) scoring instead of accuracy — on
a ~1.7% positive rate, accuracy is misleading. The best parameters found by the
search are used directly in the final model (no hardcoded overrides, all
parameters stay within the searched grid).

In [ ]:
parameters = {
    'max_depth': [5, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1],
    'reg_lambda': [1, 1.5, 3],
    'n_estimators': [300, 500, 700]
}

model_xgb = XGBClassifier(random_state=42)

# average_precision == PR-AUC: the right objective for severe class imbalance.
grid_search = GridSearchCV(model_xgb, parameters, cv=3, scoring='average_precision')
grid_search.fit(model_train_data, y_train)

best_params = grid_search.best_params_
print(f"Best parameters: {best_params}")

In [ ]:
# Use the parameters found by the grid search — no hardcoded overrides, all
# values stay within the searched grid.
model_xgb = XGBClassifier(**best_params, random_state=42)
model_xgb.fit(model_train_data, y_train)
model_xgb

## Testing

In [ ]:
y_pred = model_xgb.predict(model_test_data)
y_score = model_xgb.predict_proba(model_test_data)[:, 1]
evaluateModel(y_test, y_pred, y_score=y_score)

## Explaining

In [ ]:
explainer = dx.Explainer(model_xgb, model_train_data, y_train)

In [ ]:
explainer.model_parts().plot()

In [ ]:
instance = model_train_data.iloc[[0]]
instance

In [ ]:
shap_values = explainer.predict_parts(instance)

In [ ]:
shap_values.plot()